# 02 - Preprocessing
Subset, regrid, normalize CMEMS and AIS data to the WPS ConvLSTM model domain.

**Inputs (from Notebook 01):** `physics_raw_region.nc`, `bgc_raw_region.nc`, `ais_raw_region.parquet`

**Outputs:** `preprocessed_features.nc` (7 channels, 41x25 grid, monthly)

In [1]:
# CELL: imports, drive mount, and shared pipeline CONFIG
from google.colab import drive
import xarray as xr
import pandas as pd
import numpy as np

drive.mount('/content/drive')

CONFIG = {
    # ------------------------------------------------------------------
    # Spatial
    # ------------------------------------------------------------------
    # Broad regional bbox: matches the cache written by Notebook 01.
    'bbox_regional': {
        'lat_min': 0,   'lat_max': 30,
        'lon_min': 110, 'lon_max': 140,
    },
    # WPS model target bbox: all ConvLSTM inputs are subset to this region.
    # Narrow domain -> higher positive-pixel density -> better F1 score.
    'bbox_model': {
        'lat_min': 10,  'lat_max': 20,
        'lon_min': 114, 'lon_max': 120,
    },

    # ------------------------------------------------------------------
    # Temporal
    # ------------------------------------------------------------------
    # Full date range for raw CMEMS/AIS load (Notebook 01).
    'date_full': {'start': '2014-01-01', 'end': '2024-12-31'},
    # ConvLSTM training window: starts 2017 for denser AIS coverage.
    'date_model': {'start': '2017-01-01', 'end': '2024-12-31'},

    # ------------------------------------------------------------------
    # Paths
    # ------------------------------------------------------------------
    'data_dir': '/content/drive/MyDrive/fishing_project/',
    'files': {
        # Source CMEMS NetCDF files (placed in data_dir by the user)
        'physics_w_nc':  'cmems_mod_glo_phy_my_0.083deg_P1M-m_1779636039565.nc',
        'physics_ht_nc': 'cmems_mod_glo_phy_my_0.083deg_P1M-m_1779635380319.nc',
        'bgc_src_nc':    'cmems_mod_glo_bgc_my_0.25deg_P1M-m_1779635372583.nc',
        # Notebook 01 outputs -> Notebook 02 inputs
        'physics_nc':    'physics_raw_region.nc',
        'bgc_nc':        'bgc_raw_region.nc',
        'ais_parquet':   'ais_raw_region.parquet',
        'ais_csv_gz':    'ais_raw_region.csv.gz',
        # Notebook 02 outputs -> Notebook 03 inputs
        'ais_gridded_nc':  'ais_fishing_effort_gridded.nc',
        'preprocessed_nc': 'preprocessed_features.nc',
    },

    # ------------------------------------------------------------------
    # AIS
    # ------------------------------------------------------------------
    'ais_use_cols': ['date', 'cell_ll_lat', 'cell_ll_lon', 'fishing_hours'],

    # ------------------------------------------------------------------
    # Depth selection for CMEMS variables
    # ------------------------------------------------------------------
    'physics_surface_depth': 0.49,   # metres, nearest-neighbour selection
    'bgc_depth_range': (0.51, 5.14), # metres, averaged over this range

    # ------------------------------------------------------------------
    # Preprocessing
    # ------------------------------------------------------------------
    'norm_method': 'minmax',
    'resample_freq': '1ME',
    'train_frac':    0.70
}

DATA_DIR = CONFIG['data_dir']
bbox_m   = CONFIG['bbox_model']
dates_m  = CONFIG['date_model']
print(f'DATA_DIR   : {DATA_DIR}')
print(f'Model bbox : {bbox_m}')
print(f'Model dates: {dates_m}')

Mounted at /content/drive
DATA_DIR   : /content/drive/MyDrive/fishing_project/
Model bbox : {'lat_min': 10, 'lat_max': 20, 'lon_min': 114, 'lon_max': 120}
Model dates: {'start': '2017-01-01', 'end': '2024-12-31'}


## STEP 1: Load and Subset Datasets

In [2]:
# CELL: load regional datasets from Notebook 01, subset to WPS model bbox and dates
bm = CONFIG['bbox_model']
dm = CONFIG['date_model']
f  = CONFIG['files']

# Load the pre-merged regional datasets saved by Notebook 01
physics_ds = xr.open_dataset(DATA_DIR + f['physics_nc'])
bgc_ds     = xr.open_dataset(DATA_DIR + f['bgc_nc'])

phys_depths = physics_ds.depth.values[:5]
bgc_depths  = bgc_ds.depth.values[:5]
print(f'Available physics depths : {phys_depths}')
print(f'Available BGC depths     : {bgc_depths}')

# Physics: select nearest surface depth, then subset to WPS model bbox and dates
surf_depth = CONFIG['physics_surface_depth']
physics_subset = physics_ds.sel(
    depth=surf_depth, method='nearest'
).sel(
    latitude=slice(bm['lat_min'], bm['lat_max']),
    longitude=slice(bm['lon_min'], bm['lon_max']),
    time=slice(dm['start'], dm['end'])
)

# BGC: average over near-surface depths, then subset to WPS model bbox and dates
d_lo, d_hi = CONFIG['bgc_depth_range']
bgc_subset = bgc_ds.sel(
    depth=slice(d_lo, d_hi),
    latitude=slice(bm['lat_min'], bm['lat_max']),
    longitude=slice(bm['lon_min'], bm['lon_max']),
    time=slice(dm['start'], dm['end'])
).mean(dim='depth')

phys_dims = dict(physics_subset.sizes)
bgc_dims  = dict(bgc_subset.sizes)
print(f'Physics subset (0.083 deg) : {phys_dims}')
print(f'BGC subset    (0.25 deg)  : {bgc_dims}')

Available physics depths : [0.494025 1.541375 2.645669 3.819495 5.078224]
Available BGC depths     : [0.50576   1.5558553 2.6676817 3.8562799 5.1403613]
Physics subset (0.083 deg) : {'time': 96, 'latitude': 121, 'longitude': 72}
BGC subset    (0.25 deg)  : {'time': 96, 'latitude': 41, 'longitude': 25}


## STEP 2: Monthly Aggregation

In [3]:
# CELL: resample to monthly means
# CMEMS data is already monthly, but resampling ensures consistent time-stamp
# labels and suppresses the FutureWarning from the deprecated '1M' string.
# CONFIG['resample_freq'] = '1ME' is the modern replacement.
freq = CONFIG['resample_freq']

physics_monthly = physics_subset.resample(time=freq).mean()
bgc_monthly     = bgc_subset.resample(time=freq).mean()

print(f'Monthly physics : {dict(physics_monthly.sizes)}')
print(f'Monthly BGC     : {dict(bgc_monthly.sizes)}')

Monthly physics : {'time': 96, 'latitude': 121, 'longitude': 72}
Monthly BGC     : {'time': 96, 'latitude': 41, 'longitude': 25}


## STEP 3: Define Target Grid and Regrid

In [4]:
# CELL: regrid physics (0.083 deg) onto the BGC native grid (0.25 deg)
# Using BGC's native coordinates as the canonical target grid so all
# 7 ConvLSTM input channels share the same spatial resolution and extent.
target_lats = bgc_monthly.latitude.values
target_lons = bgc_monthly.longitude.values

n_lat     = len(target_lats)
n_lon     = len(target_lons)
lat_range = (target_lats.min(), target_lats.max())
lon_range = (target_lons.min(), target_lons.max())
print(f'Target grid : {n_lat} x {n_lon} at 0.25 deg resolution')
print(f'  Lat range : {lat_range[0]:.2f} to {lat_range[1]:.2f}')
print(f'  Lon range : {lon_range[0]:.2f} to {lon_range[1]:.2f}')

# Bilinear interpolation of physics to match BGC grid
physics_regrid = physics_monthly.interp(
    latitude=target_lats,
    longitude=target_lons,
    method='linear'
)

print(f'Physics regridded : {dict(physics_regrid.sizes)}')

Target grid : 41 x 25 at 0.25 deg resolution
  Lat range : 10.00 to 20.00
  Lon range : 114.00 to 120.00
Physics regridded : {'time': 96, 'latitude': 41, 'longitude': 25}


## STEP 4: Verify Alignment

In [5]:
# CELL: assert grid and time alignment before downstream processing
assert np.allclose(physics_regrid.latitude,  bgc_monthly.latitude),  'Latitude mismatch!'
assert np.allclose(physics_regrid.longitude, bgc_monthly.longitude), 'Longitude mismatch!'
assert len(physics_regrid.time) == len(bgc_monthly.time),            'Time-step count mismatch!'

n_months = len(bgc_monthly.time)
n_pixels = n_lat * n_lon
print(f'All grids aligned -- {n_months} months at 0.25 deg resolution')
print(f'  Grid size : {n_lat} lat x {n_lon} lon = {n_pixels:,} pixels/month')

All grids aligned -- 96 months at 0.25 deg resolution
  Grid size : 41 lat x 25 lon = 1,025 pixels/month


## STEP 5: Gap Filling

In [7]:
# CELL: fill NaN gaps with linear temporal interpolation
def fill_gaps(data):
    filled = data.interpolate_na(dim='time', method='linear', fill_value='extrapolate')
    # Clamp to valid physical range after extrapolation at edges
    # xarray.Dataset.clip() accepts Dataset objects directly to clip per-variable
    return filled.clip(min=data.min(), max=data.max())

physics_filled = fill_gaps(physics_regrid)
bgc_filled     = fill_gaps(bgc_monthly)

# Report any remaining NaNs per variable after filling
for ds_name, ds in [('physics', physics_filled), ('bgc', bgc_filled)]:
    for var in ds.data_vars:
        n_nan  = int(np.isnan(ds[var].values).sum())
        status = 'OK' if n_nan == 0 else f'WARNING {n_nan} NaNs remain'
        print(f'  {ds_name}.{var}: {status}')

print('Gap filling complete.')


  physics.uo: WARNING 9024 NaNs remain
  physics.vo: WARNING 9024 NaNs remain
  physics.zos: WARNING 9024 NaNs remain
  physics.thetao: WARNING 9024 NaNs remain
  bgc.chl: WARNING 1536 NaNs remain
  bgc.nppv: WARNING 1536 NaNs remain
Gap filling complete.


## STEP 6: Extract and Normalize Variables

In [8]:
# CELL: extract all 6 oceanographic channels and normalize
# Normalization stats are computed from training months only (no data leakage).
# fit_slice indexes the first train_frac proportion of months.

def normalize(data, method=None, fit_slice=None):
    """Normalize a DataArray. fit_slice restricts stat computation to training months."""
    if method is None:
        method = CONFIG['norm_method']
    ref = data.isel(time=fit_slice) if fit_slice is not None else data
    if method == 'minmax':
        mn, mx = float(ref.min()), float(ref.max())
        return (data - mn) / (mx - mn) if mx > mn else data * 0
    elif method == 'zscore':
        return (data - ref.mean()) / ref.std()
    raise ValueError(f'Unknown normalization method: {method}')

n_months_total = len(bgc_filled.time)
train_end      = int(CONFIG['train_frac'] * n_months_total)
fit_sl         = slice(0, train_end)

print(f'Normalization fit window : months 0–{train_end-1} of {n_months_total} total')

chl  = normalize(bgc_filled['chl'],           fit_slice=fit_sl)
nppv = normalize(bgc_filled['nppv'],          fit_slice=fit_sl)
ssh  = normalize(physics_filled['zos'],       fit_slice=fit_sl)
sst  = normalize(physics_filled['thetao'],    fit_slice=fit_sl)
uo   = normalize(physics_filled['uo'],        fit_slice=fit_sl)
vo   = normalize(physics_filled['vo'],        fit_slice=fit_sl)

method_used = CONFIG['norm_method']
print(f'Normalization complete (method={method_used})')
for var_name, arr in [('Chl', chl), ('NPPV', nppv), ('SSH', ssh),
                       ('SST', sst), ('UO',  uo),   ('VO',  vo)]:
    arr_min = float(arr.min())
    arr_max = float(arr.max())
    print(f'  {var_name:4s}: shape={arr.shape}  min={arr_min:.3f}  max={arr_max:.3f}')

Normalization fit window : months 0–66 of 96 total
Normalization complete (method=minmax)
  Chl : shape=(96, 41, 25)  min=0.000  max=1.000
  NPPV: shape=(96, 41, 25)  min=-0.000  max=1.000
  SSH : shape=(96, 41, 25)  min=0.000  max=1.063
  SST : shape=(96, 41, 25)  min=0.000  max=1.024
  UO  : shape=(96, 41, 25)  min=0.000  max=1.018
  VO  : shape=(96, 41, 25)  min=0.000  max=1.090


## STEP 7: Process AIS Data

In [9]:
# CELL: load pre-filtered AIS Parquet from Notebook 01, subset to WPS model bbox
# and date window, aggregate to 0.25 deg monthly grid, log-normalise, cache as NetCDF.
# NOTE: load_ais_filtered() is NOT redefined here -- Notebook 01 already filtered
# and saved ais_raw_region.parquet. We simply load and narrow that cache.
bm = CONFIG['bbox_model']
dm = CONFIG['date_model']
f  = CONFIG['files']

# --- 7a: Load from Notebook 01 output (Parquet preferred, CSV.gz fallback) ---
ais_parquet_name = f['ais_parquet']
ais_csv_gz_name  = f['ais_csv_gz']
print('Loading pre-filtered AIS data from Notebook 01...')
try:
    ais_df = pd.read_parquet(DATA_DIR + ais_parquet_name)
    print(f'  Loaded Parquet : {ais_parquet_name}')
except Exception:
    ais_df = pd.read_csv(DATA_DIR + ais_csv_gz_name)
    print(f'  Loaded CSV.gz fallback : {ais_csv_gz_name}')

# --- 7b: Narrow the regional AIS cache to the WPS model bbox ---
mask = (
    (ais_df['cell_ll_lat'] >= bm['lat_min']) & (ais_df['cell_ll_lat'] <  bm['lat_max']) &
    (ais_df['cell_ll_lon'] >= bm['lon_min']) & (ais_df['cell_ll_lon'] <  bm['lon_max'])
)
ais_df = ais_df[mask].copy()
ais_df['date']       = pd.to_datetime(ais_df['date'])
ais_df['year_month'] = ais_df['date'].dt.to_period('M')

# Further filter to the model date window (2017-2024)
date_mask = (
    (ais_df['date'] >= dm['start']) &
    (ais_df['date'] <= dm['end'])
)
ais_df = ais_df[date_mask]

n_records    = len(ais_df)
ais_min_date = ais_df['date'].min().date()
ais_max_date = ais_df['date'].max().date()
n_months_ais = ais_df['year_month'].nunique()
print(f'AIS records in WPS model bbox : {n_records:,}')
print(f'Date range                    : {ais_min_date} to {ais_max_date}')
print(f'Unique months                 : {n_months_ais}')


# --- 7c: Aggregate 0.1 deg GFW rows -> 0.25 deg monthly grid ---
def aggregate_ais_to_grid(ais_df, target_lats, target_lons):
    '''Bin AIS fishing_hours into the 0.25 deg target grid per month.
    GFW lower-left cell corners are shifted +0.05 to get cell centres.'''
    lat_c = ais_df['cell_ll_lat'].values + 0.05
    lon_c = ais_df['cell_ll_lon'].values + 0.05

    lat_idx = (np.searchsorted(target_lats, lat_c) - 1).clip(0, len(target_lats) - 1)
    lon_idx = (np.searchsorted(target_lons, lon_c) - 1).clip(0, len(target_lons) - 1)

    df = ais_df.copy()
    df['lat_idx'] = lat_idx
    df['lon_idx'] = lon_idx

    result = {}
    for period, grp in df.groupby('year_month'):
        grid = np.zeros((len(target_lats), len(target_lons)), dtype=np.float32)
        np.add.at(grid,
                  (grp['lat_idx'].values, grp['lon_idx'].values),
                  grp['fishing_hours'].values)
        result[str(period)] = grid
    return result


def ais_to_xarray(ais_grids, cmems_times, target_lats, target_lons):
    '''Align monthly AIS grids to the CMEMS time axis; missing months filled with 0.'''
    data = np.zeros((len(cmems_times), len(target_lats), len(target_lons)), dtype=np.float32)
    for i, t in enumerate(cmems_times):
        key = str(pd.Timestamp(t).to_period('M'))
        if key in ais_grids:
            data[i] = ais_grids[key]
    return xr.DataArray(
        data,
        dims=['time', 'latitude', 'longitude'],
        coords={'time': cmems_times, 'latitude': target_lats, 'longitude': target_lons},
        name='fishing_hours'
    )


def normalize_ais(da, fit_slice=None):
    """Log1p then min-max normalization, fitted on training months only."""
    log_da = np.log1p(da)
    ref    = log_da.isel(time=fit_slice) if fit_slice is not None else log_da
    mn, mx = float(ref.min()), float(ref.max())
    return (log_da - mn) / (mx - mn) if mx > mn else log_da * 0

ais_grids      = aggregate_ais_to_grid(ais_df, target_lats, target_lons)
ais_da         = ais_to_xarray(ais_grids, bgc_monthly.time.values, target_lats, target_lons)
ais_normalized = normalize_ais(ais_da, fit_slice=fit_sl)  # ← pass fit_sl here

n_months_agg = len(ais_grids)
ais_da_sizes = dict(ais_da.sizes)
ais_norm_min = float(ais_normalized.min())
ais_norm_max = float(ais_normalized.max())
print(f'Months aggregated : {n_months_agg}')
print(f'AIS DataArray     : {ais_da_sizes}')
print(f'AIS normalized    : min={ais_norm_min:.3f}  max={ais_norm_max:.3f}')

# Cache gridded (un-normalised) AIS to Drive -- skips re-aggregation on restart
ais_gridded_nc_name = f['ais_gridded_nc']
ais_da.to_netcdf(DATA_DIR + ais_gridded_nc_name)
print(f'Saved AIS grid : {ais_gridded_nc_name}')

Loading pre-filtered AIS data from Notebook 01...
  Loaded Parquet : ais_raw_region.parquet
AIS records in WPS model bbox : 142,595
Date range                    : 2017-01-01 to 2024-12-01
Unique months                 : 96
Months aggregated : 96
AIS DataArray     : {'time': 96, 'latitude': 41, 'longitude': 25}
AIS normalized    : min=0.000  max=1.337
Saved AIS grid : ais_fishing_effort_gridded.nc


In [10]:
monthly_effort = ais_da.sum(dim=['latitude', 'longitude']).to_series()
print("Total fishing hours per year:")
print(monthly_effort.resample('YE').sum().round(0))

# Positive pixel ratio per year (cells > 0 as fraction of all cells)
# This tells you if label density is consistent across years.
# Sudden drops → sparse AIS coverage. Sudden jumps → detection improvement.
n_pixels = n_lat * n_lon
binary_da = (ais_da > 0).astype(float)
monthly_pos = binary_da.sum(dim=['latitude', 'longitude']).to_series()
yearly_pos  = (monthly_pos / n_pixels).resample('YE').mean()
print("\nMean positive pixel ratio per year (fraction of grid cells with any fishing):")
print(yearly_pos.round(4))

Total fishing hours per year:
time
2017-12-31         0.0
2018-12-31        45.0
2019-12-31        29.0
2020-12-31        13.0
2021-12-31      1169.0
2022-12-31     54709.0
2023-12-31    136529.0
2024-12-31    205277.0
Freq: YE-DEC, Name: fishing_hours, dtype: float32

Mean positive pixel ratio per year (fraction of grid cells with any fishing):
time
2017-12-31    0.0000
2018-12-31    0.0004
2019-12-31    0.0004
2020-12-31    0.0002
2021-12-31    0.0122
2022-12-31    0.2127
2023-12-31    0.3025
2024-12-31    0.3447
Freq: YE-DEC, Name: fishing_hours, dtype: float64


## STEP 8: Save Preprocessed Data

In [11]:
# CELL: combine all 7 channels into one xarray Dataset and save to NetCDF
# This file is the sole input to 03_model_training.ipynb.
# Save normalization stats alongside the preprocessed NetCDF
# These are needed to inverse-transform predictions in 04_evaluation.
import json

norm_stats = {}
for var_name, da_raw, da_norm in [
    ('chl',            bgc_filled['chl'],        chl),
    ('nppv',           bgc_filled['nppv'],       nppv),
    ('ssh',            physics_filled['zos'],    ssh),
    ('sst',            physics_filled['thetao'], sst),
    ('uo',             physics_filled['uo'],     uo),
    ('vo',             physics_filled['vo'],     vo),
]:
    ref = da_raw.isel(time=fit_sl)
    norm_stats[var_name] = {
        'min': float(ref.min()),
        'max': float(ref.max()),
        'method': CONFIG['norm_method'],
    }

# AIS log1p stats
log_ais_ref = np.log1p(ais_da.isel(time=fit_sl))
norm_stats['fishing_effort'] = {
    'log1p_min': float(log_ais_ref.min()),
    'log1p_max': float(log_ais_ref.max()),
    'method': 'log1p_minmax',
}

norm_stats_path = DATA_DIR + 'norm_stats.json'
with open(norm_stats_path, 'w') as fh:
    json.dump(norm_stats, fh, indent=2)
print(f'Saved normalization stats : norm_stats.json')

# Then your existing save block continues unchanged:
preprocessed = xr.Dataset({
    'chl':            chl,
    'nppv':           nppv,
    'ssh':            ssh,
    'sst':            sst,
    'uo':             uo,
    'vo':             vo,
    'fishing_effort': ais_normalized,
})
# ... rest unchanged

preprocessed_nc_name = CONFIG['files']['preprocessed_nc']
out_path = DATA_DIR + preprocessed_nc_name
preprocessed.to_netcdf(out_path)

final_dims = dict(preprocessed.sizes)
final_vars = list(preprocessed.data_vars)
print(f'Preprocessed data saved : {preprocessed_nc_name}')
print(f'  Dimensions : {final_dims}')
print(f'  Variables  : {final_vars}')
print('Notebook 02 complete. Run 03_model_training.ipynb next.')

Saved normalization stats : norm_stats.json
Preprocessed data saved : preprocessed_features.nc
  Dimensions : {'latitude': 41, 'longitude': 25, 'time': 96}
  Variables  : ['chl', 'nppv', 'ssh', 'sst', 'uo', 'vo', 'fishing_effort']
Notebook 02 complete. Run 03_model_training.ipynb next.
